# Mini-samples of CloudSEN12, Sen1Floods11 and THRawS
Downloads **a few samples only** of each dataset (a few tens of MB in total), prints their shapes and plots them.
Run the cells in order. Untested paths are marked `CHECK` in the comments: if a cell fails, paste me the error.

In [ ]:
!pip -q install tacoreader rasterio remotezip requests matplotlib

In [ ]:
import os, requests, numpy as np, matplotlib.pyplot as plt, rasterio
os.makedirs("samples", exist_ok=True)

def show_rgb(arr, bands=(3,2,1), title="", scale=3000):
    # arr: (C, H, W) Sentinel-2; B04,B03,B02 = indices 3,2,1 in the 13-band order
    rgb = np.stack([arr[b] for b in bands], -1).astype("float32")
    return np.clip(rgb / scale, 0, 1)

## 1. Sen1Floods11 (public Google Cloud bucket, plain HTTP)
Lists the hand-labelled Sentinel-2 chips and downloads 3 of them (about 7 MB each) with their water masks.

In [ ]:
BUCKET = "sen1floods11"
PREFIX = "v1.1/data/flood_events/HandLabeled/"   # CHECK: folder layout

def list_objects(prefix, n=200):
    url = f"https://storage.googleapis.com/storage/v1/b/{BUCKET}/o"
    r = requests.get(url, params={"prefix": prefix, "maxResults": n}, timeout=60)
    r.raise_for_status()
    return [o["name"] for o in r.json().get("items", [])]

s2_files = [f for f in list_objects(PREFIX + "S2Hand/") if f.endswith(".tif")]
print(len(s2_files), "S2 chips listed, e.g.", s2_files[:2])

def download(name, out):
    url = f"https://storage.googleapis.com/{BUCKET}/{name}"
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(out, "wb") as f:
            for chunk in r.iter_content(1 << 20): f.write(chunk)
    return out

chips = []
for name in s2_files[:3]:
    lab = name.replace("S2Hand/", "LabelHand/").replace("_S2Hand", "_LabelHand")
    chips.append((download(name, "samples/" + os.path.basename(name)),
                  download(lab,  "samples/" + os.path.basename(lab))))

fig, ax = plt.subplots(2, 3, figsize=(10, 6))
for i, (s2p, lbp) in enumerate(chips):
    with rasterio.open(s2p) as s: x = s.read()          # (13, 512, 512), UInt16, reflectance x 10000
    with rasterio.open(lbp) as l: y = l.read(1)         # -1 no data, 0 not water, 1 water
    valid = (y != -1).sum()
    frac = float((y == 1).sum() / valid) if valid else float("nan")
    print(os.path.basename(s2p), x.shape, x.dtype, "| water fraction:", round(frac, 3))
    ax[0, i].imshow(show_rgb(x)); ax[0, i].set_title("S2 RGB"); ax[0, i].axis("off")
    ax[1, i].imshow(y, vmin=-1, vmax=1, cmap="coolwarm"); ax[1, i].set_title("label (-1/0/1)"); ax[1, i].axis("off")
plt.tight_layout(); plt.show()

## 2. CloudSEN12 (TACO format: reads only the chosen samples, not the whole dataset)

In [ ]:
import tacoreader
ds = tacoreader.load("tacofoundation:cloudsen12-l1c")      # CHECK: name; alternatives: cloudsen12-l2a, cloudsen12-extra
print(ds.columns.tolist()[:20]); print(len(ds), "samples")

# Keep only high-quality labels if the column exists (CHECK column name)
sub = ds
if "label_type" in ds.columns:
    sub = ds[ds["label_type"] == "high"]
print(len(sub), "high-quality samples")

picked = [0, len(sub) // 2, len(sub) - 1]                  # 3 samples
fig, ax = plt.subplots(2, 3, figsize=(10, 6))
for j, k in enumerate(picked):
    s = sub.read(k)
    img_path, lab_path = s.read(0), s.read(1)              # CHECK: 0 = S2 image, 1 = label (GDAL /vsisubfile paths)
    with rasterio.open(img_path) as a: x = a.read()
    with rasterio.open(lab_path) as b: y = b.read(1)
    print(k, x.shape, x.dtype, "| classes in label:", np.unique(y))
    ax[0, j].imshow(show_rgb(x)); ax[0, j].set_title("S2 L1C RGB"); ax[0, j].axis("off")
    ax[1, j].imshow(y, vmin=0, vmax=3, cmap="viridis"); ax[1, j].set_title("0 clear,1 thick,2 thin,3 shadow", fontsize=8); ax[1, j].axis("off")
plt.tight_layout(); plt.show()

## 3. THRawS (raw Level-0, 17 zips of about 9 GB each on Zenodo)
We do **not** download a zip: `remotezip` reads the zip directory over HTTP and extracts only the smallest files.
Decoding the raw granules into images needs the PyRawS library, so here we only inspect the contents.

In [ ]:
from remotezip import RemoteZip
rec = requests.get("https://zenodo.org/api/records/7908728", timeout=60).json()
files = rec["files"]                                        # CHECK: key may be "files" with "key"/"links"
print(len(files), "files on Zenodo")
for f in files[:5]: print(f.get("key"), round(f["size"] / 1e9, 2), "GB")

smallest = min((f for f in files if f["key"].endswith(".zip")), key=lambda f: f["size"])
url = smallest["links"]["self"]
print("using", smallest["key"])

with RemoteZip(url, support_suffix_range=False) as z:   # Zenodo may not accept suffix byte ranges
    infos = z.infolist()
    print(len(infos), "entries; first ones:")
    for i in infos[:10]: print(" ", i.filename, i.file_size)
    small = sorted([i for i in infos if not i.is_dir()], key=lambda i: i.file_size)[:3]
    for i in small:
        z.extract(i, "samples/thraws"); print("extracted", i.filename, i.file_size, "bytes")

## Next
- Sen1Floods11 and CloudSEN12 samples are normal GeoTIFFs: cut `x[:, i:i+16, j:j+16]` and compute `rho` as in Section "From masks and boxes to patch labels".
- THRawS: install PyRawS (`pip install git+https://github.com/ESA-PhiLab/PyRawS`) and follow its notebooks to decode granules and read the bounding boxes.